[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03a_baseline.ipynb)

# 03a — Baselines: random search and a rule-based sweep

**Purpose.** Establish what the tilt space gives up without a model, so the
Bayesian result in `03b_mobo.ipynb` has something honest to be measured against.

**Inputs.** The committed cell table in `configs/simulation.yaml`, the scenario
manifest, and `data/processed/mdt.parquet` for the band priority score.

**Outputs.** `outputs/optim/random/<timestamp>/` and
`outputs/optim/rule/<timestamp>/`, each holding the same artifacts 03b writes.

---

### The two baselines

**Random search** draws Sobol points over the same box at the **same evaluation
budget** as the Bayesian run, through the same loop — only the generation
strategy differs. That makes it a control on the model rather than a separate
experiment: any gap between it and 03b is what the surrogate bought.

**The rule-based sweep** is the operator heuristic: every cell on a band points
alike, collapsing 36 dimensions to 3, swept by coordinate descent. It is not
budget-matched and is not meant to be. It answers a different question — how
much of the achievable gain needs per-cell freedom at all.

Both score through `src/kpi/` and pick their winner with the same ADR 0001
single pass 03b uses, so the three runs differ only in where they look.


## 0. Environment

Run this section first, wherever you are.

**Locally** it walks up to the project root and makes it the working directory,
so the root-relative paths in the configs resolve the way they do for
`task bo` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path`, and installs
what Colab does not ship. Sionna-RT needs a CUDA GPU — select a GPU runtime
first, or every evaluation will fail.


In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook in this project.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab ships numpy, pandas, pyarrow, matplotlib and
# seaborn; these are the ones it does not. sionna-rt needs a CUDA runtime, so
# select a GPU runtime before running this notebook.
COLAB_PACKAGES = [
    ("hydra", "hydra-core"),
    ("sionna.rt", "sionna-rt"),
    ("ax", "ax-platform"),
]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR if SUBDIR else checkout
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Reading them
# from the environment lets a smoke run shrink the budget, or a Colab session
# repoint the paths, without editing the notebook.
CONFIG_OVERRIDES: list[str] = os.environ.get("BAND_TILT_OVERRIDES", "").split()

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ is DVC-tracked and not part of the clone, so a fresh Colab runtime has
# no scenario to optimize against. Either run notebook 00 first, or mount Drive
# and point the config at a copy that already holds one. Drive also survives a
# runtime reset, which /content does not.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# PROJECT_ROOT = "/content/drive/MyDrive/band-tilt"
# CONFIG_OVERRIDES += [
#     f"simulation.output.manifest_file={PROJECT_ROOT}/data/external/scenario.json",
#     f"data.output.mdt_file={PROJECT_ROOT}/data/processed/mdt.parquet",
#     f"bo.output.dir={PROJECT_ROOT}/outputs/optim",
# ]

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.utils.plotting import save_fig, setup_plotting
from src.utils.seed import set_seed

# Ax logs every generated trial at INFO, which here means all 36 tilts per
# trial. Warnings still come through - short batches are worth seeing.
logging.getLogger("ax").setLevel(logging.WARNING)

cfg = load_config(overrides=["bo.method=random", *CONFIG_OVERRIDES])
set_seed(cfg.seed)
setup_plotting()

# Figures land under reports/ so they can be looked at again without rerunning
# the search. Skipped on Colab, where /content does not survive a reset.
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/03a_baseline"))

pd.set_option("display.max_columns", 60)
print(f"method={cfg.bo.method}  seed={cfg.bo.seed}")
print(f"budget: n_init={cfg.bo.budget.n_init} n_iter={cfg.bo.budget.n_iter} "
      f"batch={cfg.bo.budget.batch_size}   rule: {cfg.bo.rule.n_steps} steps "
      f"x {cfg.bo.rule.n_rounds} rounds")

## 2. The search space

One absolute tilt per **(cell, band)** pair. The bounds are the committed ones
in `configs/simulation.yaml`; nothing here may widen them, and
`TiltSpace.to_cells` raises rather than ray-tracing a proposal that leaves the
box.


In [ ]:
from src.optim.space import TiltSpace

space = TiltSpace.from_config(cfg)
print(f"{space.n_dim} decision variables: {len(space.cells)} cells x {len(space.band_names)} bands")

bounds = space.as_frame(space.baseline)
bounds.groupby("band", observed=True).agg(
    baseline=("tilt_deg", "first"),
    low=("tilt_min_deg", "first"),
    high=("tilt_max_deg", "first"),
    n_cells=("cell", "size"),
)

## 3. The incumbent

Everything is measured against the tilts currently committed in
`configs/simulation.yaml`. Evaluating them first does three jobs: it is the
reference row of every comparison, it anchors Ax's objective thresholds, and it
is the check that this evaluator agrees with the simulation stage — these five
numbers must match the ones `data/interim/radio_map.npz` scores to.


In [ ]:
from src.kpi.serving import max_rsrp
from src.optim.evaluator import Evaluator
from src.optim.objective import KPI_NAMES


def kpi_table(named):
    """One column per configuration, one row per KPI, in priority order."""
    return pd.DataFrame({name: kpi.as_dict() for name, kpi in named.items()}).loc[list(KPI_NAMES)]

In [ ]:
evaluator = Evaluator(cfg, keep_rsrp=True)
print(f"scenario {evaluator.scenario_id}   bands {evaluator.band_labels}")

incumbent = evaluator.evaluate(space.baseline)
print(f"one evaluation = {incumbent.seconds:.1f}s of ray tracing\n")
kpi_table({"incumbent": incumbent.kpi})

## 4. Both baselines

Neither fits a model, so both are pure ray tracing — roughly 30-40 s per
candidate and nothing else. That makes random search cheaper than 03b at the
same evaluation count, but not dramatically so: the gap is the model's cost, and
it is recorded below because a comparison matched on evaluations and unmatched
on wall clock has to say both numbers.


In [ ]:
import time

from src.optim.search import run_search

runs = {}
for method in ("random", "rule"):
    started = time.time()
    runs[method] = run_search(evaluator, cfg, method)
    elapsed = time.time() - started
    frame = runs[method].frame()
    print(f"{method:>7}: {len(runs[method]):3d} evaluations, "
          f"{int(frame['on_pareto'].sum()):3d} non-dominated, "
          f"{elapsed / 60:5.1f} min "
          f"({frame['seconds'].sum() / elapsed:.0%} of it ray tracing)")

## 5. What each baseline found

The winner of each run under the same priority order. A method whose winner is
evaluation 0 found nothing that beat the deployed tilts — which is a result, not
a failure of the notebook.


In [ ]:
winners = {"incumbent": incumbent.kpi}
best_of = {}
for method, history in runs.items():
    index = history.best_index(cfg)
    best_of[method] = index
    winners[method] = history.results[index].kpi
    print(f"{method:>7}: winner is evaluation {index} of {len(history)}")

comparison = kpi_table(winners)
comparison["direction"] = ["maximise" if n == "band_priority_score" else "minimise"
                           for n in comparison.index]
comparison

### Where they looked

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

colours = {"random": "tab:green", "rule": "tab:purple"}
for method, history in runs.items():
    f = history.frame()
    axes[0].scatter(f["hole_rate"], f["band_priority_score"], s=20,
                    alpha=0.55, c=colours[method], label=method)
axes[0].scatter([incumbent.kpi.hole_rate], [incumbent.kpi.band_priority_score],
                s=140, marker="*", c="tab:red", zorder=5, label="incumbent")
axes[0].set(xlabel="hole rate (minimise)", ylabel="band priority score (maximise)",
            title="Where each baseline looked")
axes[0].legend(fontsize=8)

# The rule sweep is one-dimensional per band, so its trace is readable directly.
rule = runs["rule"].frame()
for band in space.band_names:
    column = f"tilt_{space.cells[0].name}_{band}"
    axes[1].plot(rule.index, rule[column], marker=".", lw=1, label=band)
axes[1].set(xlabel="evaluation", ylabel="shared tilt (deg)",
            title="The rule-based sweep, band by band")
axes[1].legend(fontsize=8, title="band")
fig.tight_layout()
save_fig(fig, "where_the_baselines_looked")

## 6. Persist

Both runs, into the same layout `03b_mobo.ipynb` writes, so the three are
readable side by side.


In [ ]:
from src.optim.history import LocalRunWriter, write_run
from src.optim.run import output_directory

for method, history in runs.items():
    directory = output_directory(cfg, method)
    index = best_of[method]
    archived = evaluator.evaluate(history.results[index].tilt_deg)
    radio_map_path = evaluator.write_radio_map(directory / "best_radio_map.npz", archived)
    write_run(
        history,
        LocalRunWriter(directory),
        cfg,
        method=method,
        best_index=index,
        extra={
            "scenario_id": evaluator.scenario_id,
            "best_radio_map": str(radio_map_path),
            "notebook": "03a_baseline.ipynb",
        },
    )
    print(f"{method:>7}: {directory}")

evaluator.close()  # release the scene and its GPU memory

## 7. All methods side by side

Reads whatever runs exist under `outputs/optim/`, so this table fills in as
`03b_mobo.ipynb` is run. Compare on evaluations **and** on wall clock: the
methods are matched on the first and nowhere near matched on the second.


In [ ]:
import json
from pathlib import Path

rows = []
for path in sorted(Path(cfg.bo.output.dir).glob("*/*/run.json")):
    run = json.loads(path.read_text(encoding="utf-8"))
    rows.append({
        "method": run["method"],
        "run": path.parent.name,
        "evaluations": run["n_evaluations"],
        "pareto": run["n_pareto"],
        "minutes": round(run.get("wall_clock_seconds", float("nan")) / 60, 1),
        **{name: round(value, 4) for name, value in run["best_kpi"].items()},
    })

if not rows:
    print("no runs under", cfg.bo.output.dir)
else:
    summary = pd.DataFrame(rows).sort_values(["method", "run"])
    display(summary.reset_index(drop=True))

## 9. Handoff checklist

- [ ] The incumbent's KPIs matched `data/interim/radio_map.npz`
- [ ] Random search ran at the same `bo.budget` as the Bayesian run
- [ ] Every evaluated tilt stayed inside the committed bounds
- [ ] The five KPIs came from `src/kpi/`, with no threshold restated here
- [ ] Each winner was chosen by the ADR 0001 single pass
- [ ] Wall clock was recorded, not only the evaluation count
- [ ] Both run directories hold history, pareto, best_tilt, best_radio_map and run.json
